# RAG Course - OpenAI Embeddings + Vector Store

En esta POC vamos a practicar el flujo real de `RAG` con embeddings de OpenAI:
- elegir modelo de embedding
- convertir documentos en chunks
- generar embeddings
- construir una base vectorial simple (en memoria)
- hacer recuperacion por similitud coseno
- generar una cadena de respuesta condicionada con contexto


## 0) Setup basico de entorno

Ajusta `OPENAI_API_KEY` para usar embeddings reales.
Si no hay clave, se muestra una advertencia y no se ejecutan llamadas a OpenAI.

In [ ]:
import os
from pathlib import Path
import sys
import math

from dotenv import load_dotenv

load_dotenv()

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (parent / "src" / "sst_chatbot").exists():
        PROJECT_ROOT = parent
        break
SRC_DIR = PROJECT_ROOT / "src"
for path in (PROJECT_ROOT, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from sst_chatbot.rag_poc import build_sample_sst_documents, load_source_documents, split_documents

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SRC_DIR = {SRC_DIR}")

HAS_OPENAI_KEY = bool(os.getenv("OPENAI_API_KEY"))
if HAS_OPENAI_KEY:
    print("OPENAI_API_KEY encontrada: listo para embeddings reales")
else:
    print("Sin OPENAI_API_KEY: no se ejecutan llamados reales hasta que la configures")


## 1) Modelos de embeddings de OpenAI (resumen practico)

Hoy conviene elegir entre: 3-small para costo/velocidad y 3-large para mejor calidad.
`text-embedding-ada-002` se incluye como legacy y suele dejar de priorizarse en nuevos proyectos.

| Modelo | Dimesion base | Uso recomendado |
| --- | ---: | --- |
| text-embedding-3-small | 1536 | default para POC y produccion liviana |
| text-embedding-3-large | 3072 | cuando importa mas precision |
| text-embedding-ada-002 | 1536 | legado, compatibilidad historica |

Tip: siempre validalo contra tu necesidad de costo, precision y longitud de contexto.

In [ ]:
from typing import Any

OPENAI_EMBEDDING_MODELS = {
    "text-embedding-3-small": {"size": 1536, "label": "RAG default"},
    "text-embedding-3-large": {"size": 3072, "label": "Mayor calidad"},
    "text-embedding-ada-002": {"size": 1536, "label": "Legado"},
}

SELECTED_EMBEDDING_MODEL: Any = "text-embedding-3-small"
if SELECTED_EMBEDDING_MODEL not in OPENAI_EMBEDDING_MODELS:
    raise ValueError(f"Modelo no soportado en esta guia: {SELECTED_EMBEDDING_MODEL}")

print("Modelo elegido:", SELECTED_EMBEDDING_MODEL)
print("Dimensiones:", OPENAI_EMBEDDING_MODELS[SELECTED_EMBEDDING_MODEL]["size"])


## 2) Cargar documentos base y hacer chunking

Usamos las mismas notas internas de SST para mantener consistencia con este repositorio.


In [ ]:
from langchain_core.documents import Document

sources = build_sample_sst_documents()
documents = load_source_documents(sources)
chunks = split_documents(documents, chunk_size=220, chunk_overlap=40)

print(f"Documentos base: {len(documents)}")
print(f"Chunks para embedding: {len(chunks)}")

for i, chunk in enumerate(chunks[:3]):
    print(f"[{i}] {chunk.metadata['source_id']} | chunk={chunk.metadata['chunk_index']}")
    print(chunk.page_content[:120], "...")
    print()


## 3) Construir embeddings con OpenAI

Aca se crean vectores de texto para cada chunk.Luego esos vectores se usan como base para buscar documentos similares.

In [ ]:
from langchain_openai import OpenAIEmbeddings

if HAS_OPENAI_KEY:
    embedding_model = OpenAIEmbeddings(model=SELECTED_EMBEDDING_MODEL)
    chunk_texts = [chunk.page_content for chunk in chunks]
    chunk_vectors = embedding_model.embed_documents(chunk_texts)
    print("Embeddings calculados:", len(chunk_vectors))
    print("Dimension de vector:", len(chunk_vectors[0]) if chunk_vectors else 0)
else:
    embedding_model = None
    chunk_vectors = []
    print("No se calcularon embeddings porque no hay OPENAI_API_KEY")


## 4) Convertir en base vectorial (en memoria) y query por similitud coseno

Esto es un indice vectorial simple y transparente para entender el mecanismo.
Despues podemos reemplazarlo por FAISS, pgvector o otro backend sin cambiar la interfaz de busqueda.

In [ ]:
from typing import Optional


def cosine_similarity(a: list[float], b: list[float]) -> float:
    if not a or not b or len(a) != len(b):
        return 0.0

    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def build_vector_store(chunks: list[Document], vectors: list[list[float]]):
    if len(chunks) != len(vectors):
        raise ValueError("chunks y vectors deben tener mismo largo")

    return [
        {
            "chunk": chunk,
            "vector": vector,
        }
        for chunk, vector in zip(chunks, vectors)
    ]

def vector_search(
    query: str,
    retriever_embedding_model: Optional[OpenAIEmbeddings],
    index: list[dict],
    top_k: int = 3,
):
    if retriever_embedding_model is None:
        raise RuntimeError("Se necesita embedding model para buscar.")

    query_vec = retriever_embedding_model.embed_query(query)
    scored = []

    for item in index:
        score = cosine_similarity(query_vec, item["vector"])
        scored.append((score, item["chunk"]))

    return sorted(scored, key=lambda x: x[0], reverse=True)[:top_k]

if HAS_OPENAI_KEY:
    vector_store = build_vector_store(chunks, chunk_vectors)
    print("Vector store listo en memoria:", len(vector_store))
else:
    vector_store = []


In [ ]:
query = "Que aporta ARDS y SDD para SST?"

if HAS_OPENAI_KEY:
    top = vector_search(query, embedding_model, vector_store, top_k=2)
    print("Consulta:", query)
    for score, chunk in top:
        m = chunk.metadata
        print(f"score={score:.4f} source={m['source_id']} chunk={m['chunk_index']} title={m['title']}")
        print(chunk.page_content[:220], "...")
        print()
else:
    print("No se ejecuta busqueda vectorial sin clave")


## 5) Cadena RAG final (pregunta -> contexto -> respuesta)

Aca conectamos el retriever artesanal con un `chat model` y la prompt.
De nuevo: el objetivo de esta celda es ver el flujo, no la infra compleja de prod.

In [ ]:
from langchain_core.messages import HumanMessage
from sst_chatbot.langchain_poc import build_chat_model`r`nfrom sst_chatbot.rag_poc import build_mock_rag_chat_model
from sst_chatbot.rag_poc import build_rag_prompt

question = "Resumime como ARDS y SDD mejora el trabajo con agentes en este repo."

USE_OPENAI = HAS_OPENAI_KEY
if USE_OPENAI:
    from sst_chatbot.config import require_env
    require_env(("OPENAI_API_KEY", "LANGCHAIN_API_KEY"))
    chat_model = build_chat_model()
else:
    chat_model = build_mock_rag_chat_model()

if HAS_OPENAI_KEY:
    retrieved = vector_search(question, embedding_model, vector_store, top_k=3)
    context = "\n\n".join(
        f"[source={chunk.metadata['source_id']} chunk={chunk.metadata['chunk_index']}] {chunk.page_content[:250]}"
        for _, chunk in retrieved
    )
else:
    context = "Sin contexto vectorial por falta de clave OpenAI"

prompt = build_rag_prompt()
message = prompt.invoke({"question": question, "context": context})
response = chat_model.invoke([HumanMessage(content=message.content)]) if hasattr(message, 'content') else chat_model.invoke([HumanMessage(content=prompt.format(question=question, context=context))])
print(response.content if hasattr(response, 'content') else response)


## 6) Proximo paso: persistir como base vectorial real

Cuando quieras pasar a persistencia, este bloque logico se sustituye por un backend:
- FAISS (`faiss-cpu`) para demo local rapido
- pgvector para produccion con Postgres
- Qdrant o Weaviate para scale distribuido
- Chroma para prototipos

La interface de entrada/salida no cambia mucho: "texto de pregunta -> docs ordenados por similitud".
Esto permite cambiar solo la capa de storage y mantener la logica del agente igual.